# pooled_ols_estimates
Sanitized research code. Default input is SYNTHETIC, not paper data. Run via tools/run_models.py. See docs/MANUSCRIPT_ALIGNMENT.md.


In [ ]:

from pathlib import Path
import re

import geopandas as gpd
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf

# Public package: never search parent folders for private data.
import os
from IPython.display import display
PROJECT_ROOT = Path(os.environ['UFUG_PUBLIC_ROOT']).resolve()
DATA_PATH = Path(os.environ.get('UFUG_MODEL_DATA', str(PROJECT_ROOT / 'data/synthetic/model_cells.geojson'))).resolve()
DATA_RELATIVE = DATA_PATH
OUTPUT_DIR = Path(os.environ.get('UFUG_OUTPUT_ROOT', str(PROJECT_ROOT / 'outputs'))) / "pooled_ols_estimates"
if DATA_PATH != (PROJECT_ROOT / 'data/synthetic/model_cells.geojson').resolve() and OUTPUT_DIR.resolve().is_relative_to(PROJECT_ROOT):
    raise ValueError('Private model outputs must resolve outside Public, including redirected model directories')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


OUTCOMES = {
    "log_F_h1": "Visit frequency",
    "log_D_h1": "Visit duration",
}

PREDICTOR_LABELS = {
    "ps": "Perceptual safety",
    "pl": "Perceptual liveliness",
    "pb": "Perceptual beauty",
    "pm": "Perceptual monotony",
    "peg": "Greenness",
    "peo": "Openness",
    "pep": "Furniture",
    "peps": "Proximity to park services",
    "per": "Road connectivity",
    "pepw": "Proximity to water",
    "set": "Set",
}

PARK_LABELS = {
    "amk": "Bishan-AMK Park",
    "jurong": "Jurong Lake Gardens",
    "punggol": "Punggol Waterway Park",
}

sns.set_theme(
    context="paper",
    style="whitegrid",
    font="DejaVu Sans",
    rc={
        "axes.edgecolor": "#333333",
        "axes.labelcolor": "#111111",
        "axes.titlesize": 10,
        "axes.labelsize": 9,
        "xtick.labelsize": 8,
        "ytick.labelsize": 8,
        "legend.fontsize": 8,
        "figure.dpi": 150,
        "savefig.dpi": 600,
        "savefig.bbox": "tight",
        "savefig.pad_inches": 0.04,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "svg.fonttype": "none",
    },
)

PALETTE = {
    "Visit frequency": "#ba121f",
    "Visit duration": "#08306b",
}


In [ ]:

gdf = gpd.read_file(DATA_PATH)

for col in ["F_h", "D_h"]:
    gdf[col] = pd.to_numeric(gdf[col], errors="coerce").fillna(0)

gdf["log_F_h1"] = np.log1p(gdf["F_h"])
gdf["log_D_h1"] = np.log1p(gdf["D_h"])
gdf["park"] = gdf["park"].astype("category")
gdf["park_label"] = gdf["park"].map(PARK_LABELS).fillna(gdf["park"].astype(str))

X_COLS = ["ps", "pl", "pb", "pm", "peg", "peo", "pep", "peps", "per", "pepw", "set"]
X_COLS = [c for c in X_COLS if c in gdf.columns]

model_data = gdf.dropna(subset=X_COLS + list(OUTCOMES)).copy()
print(f"Loaded {len(gdf):,} hexagons; using {len(model_data):,} complete cases across {model_data['park'].nunique()} parks.")
print("Predictors:", ", ".join(X_COLS))

display(
    model_data.groupby("park", observed=True)
    .agg(
        n_hex=("grid_id", "count"),
        total_visits=("F_h", "sum"),
        mean_visits=("F_h", "mean"),
        total_duration=("D_h", "sum"),
        mean_duration=("D_h", "mean"),
    )
    .round(2)
)


In [ ]:

rhs = " + ".join(X_COLS) + " + C(park)"
FORMULAS = {
    "Visit frequency": f"log_F_h1 ~ {rhs}",
    "Visit duration": f"log_D_h1 ~ {rhs}",
}

models = {
    outcome_label: smf.ols(formula, data=model_data).fit(cov_type="HC3")
    for outcome_label, formula in FORMULAS.items()
}

for outcome_label, model in models.items():
    print(f"\nPooled OLS ({outcome_label})")
    print(model.summary())


In [ ]:

def clean_term(term):
    if term == "Intercept":
        return "Intercept"
    park_match = re.match(r"C\(park\)\[T\.(.+)\]", term)
    if park_match:
        park = park_match.group(1)
        return f"Park: {PARK_LABELS.get(park, park)}"
    return PREDICTOR_LABELS.get(term, term)

estimate_rows = []
fit_rows = []
prediction_frames = []

for outcome_label, model in models.items():
    conf = model.conf_int(alpha=0.05)
    for term in model.params.index:
        estimate_rows.append(
            {
                "outcome": outcome_label,
                "term": term,
                "label": clean_term(term),
                "estimate": model.params[term],
                "std_error_hc3": model.bse[term],
                "z_value": model.tvalues[term],
                "p_value": model.pvalues[term],
                "ci_low": conf.loc[term, 0],
                "ci_high": conf.loc[term, 1],
            }
        )

    fit_rows.append(
        {
            "outcome": outcome_label,
            "formula": model.model.formula,
            "n": int(model.nobs),
            "r_squared": model.rsquared,
            "adj_r_squared": model.rsquared_adj,
            "aic": model.aic,
            "bic": model.bic,
            "covariance": "HC3",
        }
    )

    pred = model_data[["grid_id", "park", "park_label", "F_h", "D_h", "log_F_h1", "log_D_h1"]].copy()
    pred["outcome"] = outcome_label
    pred["observed"] = model.model.endog
    pred["fitted"] = model.fittedvalues
    pred["residual"] = model.resid
    prediction_frames.append(pred)

coef_table = pd.DataFrame(estimate_rows)
fit_table = pd.DataFrame(fit_rows)
predictions = pd.concat(prediction_frames, ignore_index=True)

def safe_to_csv(df, filename):
    output_path = OUTPUT_DIR / filename
    try:
        df.to_csv(output_path, index=False)
        return output_path
    except PermissionError:
        fallback_path = output_path.with_name(f"{output_path.stem}_corrected{output_path.suffix}")
        df.to_csv(fallback_path, index=False)
        print(f"Skipped locked file: {output_path}; wrote {fallback_path.name}")
        return fallback_path

safe_to_csv(coef_table, "pooled_ols_coefficients.csv")
safe_to_csv(fit_table, "pooled_ols_model_fit.csv")
safe_to_csv(predictions, "pooled_ols_predictions.csv")

latex_table = coef_table.copy()
latex_table["estimate_se"] = latex_table.apply(
    lambda r: f"{r['estimate']:.3f} ({r['std_error_hc3']:.3f})",
    axis=1,
)
latex_wide = latex_table.pivot(index="label", columns="outcome", values="estimate_se")
latex_path = OUTPUT_DIR / "pooled_ols_coefficients.tex"
try:
    latex_wide.to_latex(latex_path, escape=True)
except PermissionError:
    fallback_latex_path = latex_path.with_name(f"{latex_path.stem}_corrected{latex_path.suffix}")
    latex_wide.to_latex(fallback_latex_path, escape=True)
    print(f"Skipped locked file: {latex_path}; wrote {fallback_latex_path.name}")

print("Saved outputs to", OUTPUT_DIR.name)
display(fit_table.round(3))
display(coef_table.round(4))


In [ ]:

std_data = model_data.copy()
for col in X_COLS:
    std = std_data[col].std(ddof=0)
    if std and not np.isclose(std, 0):
        std_data[col] = (std_data[col] - std_data[col].mean()) / std

std_models = {
    outcome_label: smf.ols(formula, data=std_data).fit(cov_type="HC3")
    for outcome_label, formula in FORMULAS.items()
}

std_rows = []
for outcome_label, model in std_models.items():
    conf = model.conf_int(alpha=0.05)
    for term in X_COLS:
        std_rows.append(
            {
                "outcome": outcome_label,
                "term": term,
                "label": clean_term(term),
                "estimate": model.params[term],
                "std_error_hc3": model.bse[term],
                "p_value": model.pvalues[term],
                "ci_low": conf.loc[term, 0],
                "ci_high": conf.loc[term, 1],
            }
        )

std_coef_table = pd.DataFrame(std_rows)
safe_to_csv(std_coef_table, "pooled_ols_standardized_coefficients.csv")
display(std_coef_table.round(4))


In [ ]:

def save_figure(fig, stem):
    for ext in ["png", "svg", "pdf"]:
        output_path = OUTPUT_DIR / f"{stem}.{ext}"
        try:
            fig.savefig(output_path)
        except PermissionError:
            print(f"Skipped locked file: {output_path}")

plot_df = std_coef_table.copy()
plot_df["significant"] = plot_df["p_value"] < 0.05
order = (
    plot_df.groupby("label", sort=False)["estimate"]
    .apply(lambda s: s.abs().max())
    .sort_values()
    .index.tolist()
)
plot_df["label"] = pd.Categorical(plot_df["label"], categories=order, ordered=True)

fig, ax = plt.subplots(figsize=(7.2, 4.8))

y_positions = np.arange(len(order))
offset = 0.16
x_min = plot_df["ci_low"].min()
x_max = plot_df["ci_high"].max()
x_pad = (x_max - x_min) * 0.02
for i, outcome in enumerate(OUTCOMES.values()):
    sub = plot_df[plot_df["outcome"] == outcome].sort_values("label").copy()
    y = y_positions + (i - 0.5) * 2 * offset
    sub["y"] = y
    for significant in [False, True]:
        part = sub[sub["significant"] == significant]
        if part.empty:
            continue
        style_color = PALETTE[outcome] if significant else "#b8b8b8"
        ax.errorbar(
            part["estimate"],
            part["y"],
            xerr=[part["estimate"] - part["ci_low"], part["ci_high"] - part["estimate"]],
            fmt="o",
            markersize=5.6 if significant else 4.3,
            capsize=2.5,
            elinewidth=1.2 if significant else 0.9,
            color=style_color,
            ecolor=style_color,
            markerfacecolor=PALETTE[outcome] if significant else "white",
            markeredgecolor=PALETTE[outcome] if significant else "#777777",
            markeredgewidth=1.1,
            alpha=0.98 if significant else 0.58,
            zorder=3 if significant else 2,
        )
    for row in sub[sub["significant"]].itertuples():
        ax.text(row.ci_high + x_pad, row.y, "*", color=PALETTE[outcome], va="center", ha="left", fontsize=9, fontweight="bold")

ax.axvline(0, color="#333333", linewidth=0.8, linestyle="--")
ax.set_yticks(y_positions)
ax.set_yticklabels(order)
ax.set_xlim(x_min - x_pad * 3, x_max + x_pad * 8)
ax.set_xlabel("Standardized coefficient, HC3 95% CI")
ax.set_ylabel("")
ax.set_title("Pooled OLS estimates of hexagon-level park use")
outcome_handles = [
    mpl.lines.Line2D([0], [0], marker="o", linestyle="", color=PALETTE[outcome], markerfacecolor=PALETTE[outcome], markeredgecolor=PALETTE[outcome], label=outcome)
    for outcome in OUTCOMES.values()
]
sig_handles = [
    mpl.lines.Line2D([0], [0], marker="o", linestyle="", color="#333333", markerfacecolor="#333333", markeredgecolor="#333333", label="p < 0.05"),
    mpl.lines.Line2D([0], [0], marker="o", linestyle="", color="#777777", markerfacecolor="white", markeredgecolor="#777777", label="p >= 0.05"),
]
legend_outcome = ax.legend(handles=outcome_handles, title="Outcome", frameon=False, loc="lower right")
ax.add_artist(legend_outcome)
ax.legend(handles=sig_handles, title="Significance", frameon=False, loc="upper right")
ax.grid(axis="x", color="#dddddd", linewidth=0.7)
ax.grid(axis="y", visible=False)
sns.despine(ax=ax, left=True, bottom=False)
fig.tight_layout()
save_figure(fig, "pooled_ols_standardized_coefficients")
plt.show()


In [ ]:

fit_plot_df = predictions.copy()

fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.2), sharex=False, sharey=False)
for ax, outcome in zip(axes, OUTCOMES.values()):
    sub = fit_plot_df[fit_plot_df["outcome"] == outcome]
    sns.scatterplot(
        data=sub,
        x="fitted",
        y="observed",
        hue="park_label",
        ax=ax,
        s=24,
        linewidth=0.25,
        edgecolor="white",
        alpha=0.85,
        palette=["#0072B2", "#D55E00", "#009E73"],
        legend=(outcome == "Visit duration"),
    )
    lim_min = min(sub["fitted"].min(), sub["observed"].min())
    lim_max = max(sub["fitted"].max(), sub["observed"].max())
    pad = (lim_max - lim_min) * 0.06 if lim_max > lim_min else 0.5
    ax.plot([lim_min - pad, lim_max + pad], [lim_min - pad, lim_max + pad], color="#333333", linewidth=0.8, linestyle="--")
    ax.set_xlim(lim_min - pad, lim_max + pad)
    ax.set_ylim(lim_min - pad, lim_max + pad)
    ax.set_title(outcome)
    ax.set_xlabel("Fitted log(value + 1)")
    ax.set_ylabel("Observed log(value + 1)")
    ax.grid(color="#dddddd", linewidth=0.7)
    if outcome == "Visit duration":
        ax.legend(title="Park", frameon=False, bbox_to_anchor=(1.04, 1), loc="upper left")
    elif ax.get_legend() is not None:
        ax.get_legend().remove()
    sns.despine(ax=ax)

fig.suptitle("Observed and fitted hexagon-level park use", y=1.03, fontsize=10)
fig.tight_layout()
save_figure(fig, "pooled_ols_observed_vs_fitted")
plt.show()
